In [1]:
# Cell 1: Install all required packages
!pip install streamlit langchain faiss-cpu pypdf2 pdf2image pytesseract pillow python-docx pymupdf groq psutil pdfplumber unstructured sentence-transformers langchain-community langchain-huggingface
!pip install -U langchain langchain-community langchain-text-splitters langchain-core
# Install system dependencies
!apt-get install poppler-utils -y
!apt-get install tesseract-ocr -y
!apt-get install libtesseract-dev -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libtesseract-dev is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [2]:
# Cell 2: Upload your knowledge_base files
from google.colab import files
import os
import zipfile

# # Create knowledge_base directory
# os.makedirs('knowledge_base', exist_ok=True)

# # Upload your PDF files (you can select multiple files)
# uploaded = files.upload()

# # If you have a zip file with all your documents
uploaded = files.upload()  # Upload your zip file
with zipfile.ZipFile('knowledge_base.zip', 'r') as zip_ref:
    zip_ref.extractall('')

Saving knowledge_base.zip to knowledge_base.zip


In [3]:
# Cell 3: Fixed document processing for Colab
import os
import pymupdf
from pdf2image import convert_from_path
import pytesseract
from PIL import Image, ImageEnhance
import gc

# NEW UPDATED IMPORTS
from langchain_community.document_loaders import PDFPlumberLoader, Docx2txtLoader, TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document  # Correct way to import Document/Schema

# Colab-specific paths
POPPLER_PATH = "/usr/bin"
TESSERACT_CMD = "/usr/bin/tesseract"

# Configure pytesseract
pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD

def is_scanned_pdf(file_path):
    """Check if PDF is scanned (image-based)"""
    try:
        with pymupdf.open(file_path) as doc:  # Using pymupdf instead of fitz
            if len(doc) == 0:
                return True

            # Check first few pages for text
            pages_to_check = min(3, len(doc))
            text_pages = 0

            for i in range(pages_to_check):
                text = doc[i].get_text().strip()
                if len(text) > 100:  # Reasonable text threshold
                    text_pages += 1

            return text_pages == 0  # No text = scanned PDF

    except Exception as e:
        print(f"PDF analysis error: {str(e)}")
        return True

def colab_ocr_pdf(file_path):
    """OCR processing for scanned PDFs in Colab"""
    try:
        print(f"🔍 OCR Processing: {os.path.basename(file_path)}")

        # Convert PDF to images
        images = convert_from_path(
            file_path,
            dpi=300,
            poppler_path=POPPLER_PATH,
            grayscale=True,
        )

        full_text = []
        for i, image in enumerate(images, 1):
            try:
                # OCR the image
                text = pytesseract.image_to_string(
                    image,
                    lang="eng",
                    config="--psm 6"
                )

                if text.strip():
                    full_text.append(f"Page {i}:\n{text}")
                    print(f"✅ OCR Page {i}")

                # Clean up
                del image
                gc.collect()

            except Exception as e:
                print(f"❌ Page {i} OCR failed: {str(e)}")
                continue

        if full_text:
            # Save OCR text
            ocr_text = "\n\n".join(full_text)
            print(f"✅ OCR extracted {len(full_text)} pages")
            return ocr_text
        else:
            print("❌ No text extracted via OCR")
            return None

    except Exception as e:
        print(f"❌ OCR processing failed: {str(e)}")
        return None

def colab_load_document(file_path):
    """Complete document loader for Colab"""
    if not os.path.exists(file_path):
        print(f"❌ File not found: {file_path}")
        return None

    file_ext = os.path.splitext(file_path)[1].lower()
    print(f"📄 Processing: {os.path.basename(file_path)}")

    try:
        if file_ext == ".pdf":
            # Check if scanned PDF
            if is_scanned_pdf(file_path):
                print("🖼️  Scanned PDF detected - using OCR")
                ocr_text = colab_ocr_pdf(file_path)

                if ocr_text:
                    # Create document from OCR text
                    from langchain.schema import Document
                    docs = [Document(page_content=ocr_text)]

                    splitter = CharacterTextSplitter(
                        chunk_size=800,
                        chunk_overlap=100,
                        separator="\n\n"
                    )
                    chunks = splitter.split_documents(docs)
                    print(f"✅ OCR split into {len(chunks)} chunks")
                    return chunks
                else:
                    return None
            else:
                # Text-based PDF
                print("📝 Text-based PDF - direct extraction")
                loader = PDFPlumberLoader(file_path)
                docs = loader.load()

                if docs:
                    splitter = CharacterTextSplitter(
                        chunk_size=1000,
                        chunk_overlap=150,
                        separator="\n\n"
                    )
                    chunks = splitter.split_documents(docs)
                    print(f"✅ PDF split into {len(chunks)} chunks")
                    return chunks
                return docs

        elif file_ext == ".docx":
            print("📝 Processing DOCX file")
            loader = Docx2txtLoader(file_path)
            docs = loader.load()

            if docs:
                splitter = CharacterTextSplitter(
                    chunk_size=1000,
                    chunk_overlap=150
                )
                chunks = splitter.split_documents(docs)
                print(f"✅ DOCX split into {len(chunks)} chunks")
                return chunks
            return docs

        elif file_ext == ".txt":
            print("📄 Processing TXT file")
            loader = TextLoader(file_path, encoding="utf-8")
            docs = loader.load()

            if docs:
                splitter = CharacterTextSplitter(
                    chunk_size=1000,
                    chunk_overlap=150
                )
                chunks = splitter.split_documents(docs)
                print(f"✅ TXT split into {len(chunks)} chunks")
                return chunks
            return docs

        else:
            print(f"❌ Unsupported file type: {file_ext}")
            return None

    except Exception as e:
        print(f"❌ Error loading {file_path}: {str(e)}")
        return None

In [4]:
# Cell 4: Train on all documents
def colab_train_all_documents():
    """Train on all documents in Colab"""
    knowledge_base_dir = "knowledge_base"

    if not os.path.exists(knowledge_base_dir):
        print("❌ Knowledge base directory not found!")
        return []

    # Get all files
    all_files = []
    for root, dirs, files in os.walk(knowledge_base_dir):
        for file in files:
            if file.lower().endswith(('.pdf', '.docx', '.txt')):
                full_path = os.path.join(root, file)
                all_files.append(full_path)

    print(f"📚 Found {len(all_files)} files to process")
    print("Files:", [os.path.basename(f) for f in all_files])

    all_documents = []

    for i, file_path in enumerate(all_files, 1):
        print(f"\n{'='*60}")
        print(f"🔄 Processing {i}/{len(all_files)}: {os.path.basename(file_path)}")

        docs = colab_load_document(file_path)
        if docs:
            all_documents.extend(docs)
            print(f"✅ Added {len(docs)} chunks (Total: {len(all_documents)})")
        else:
            print("⚠️ No documents extracted")

    print(f"\n🎯 TRAINING COMPLETED!")
    print(f"📊 Total documents: {len(all_documents)}")

    # Calculate total text size
    total_size = sum(len(d.page_content) for d in all_documents) / 1024 / 1024
    print(f"💾 Total text size: {total_size:.2f} MB")

    return all_documents

# Start training
print("🚀 Starting training in Google Colab...")
documents = colab_train_all_documents()

🚀 Starting training in Google Colab...
📚 Found 11 files to process
Files: ['Muslim Personal Law (Shariat) Application Act (1937).pdf', 'Consumer Protection Act (2019).pdf', 'Special Marriage Act (1954).pdf', 'Code on Wages, 2019.pdf', 'Consumer-Protection-E-Commerce-Rules-2020.pdf', 'Digital Personal Data Protection Act (2023).pdf', 'Code on Social Security, 2020.pdf', 'Indian Christian Marriage Act (1872).pdf', 'new_labour_codes-2020-.pdf', 'Information Technology Act (2000).pdf', 'Hindu Marriage Act (1955).pdf']

🔄 Processing 1/11: Muslim Personal Law (Shariat) Application Act (1937).pdf
📄 Processing: Muslim Personal Law (Shariat) Application Act (1937).pdf
📝 Text-based PDF - direct extraction
✅ PDF split into 4 chunks
✅ Added 4 chunks (Total: 4)

🔄 Processing 2/11: Consumer Protection Act (2019).pdf
📄 Processing: Consumer Protection Act (2019).pdf
📝 Text-based PDF - direct extraction
✅ PDF split into 39 chunks
✅ Added 39 chunks (Total: 43)

🔄 Processing 3/11: Special Marriage Act (1

In [5]:
# Cell 5: Create and save FAISS vector store
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

def create_vector_store(documents):
    """Create FAISS vector store"""
    if not documents:
        print("❌ No documents to create vector store")
        return None

    print("🔄 Creating embeddings...")

    # Use efficient model for Colab
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={'device': 'cpu'}
    )

    print("🔄 Creating FAISS vector store...")
    vector_store = FAISS.from_documents(documents, embeddings)

    print("💾 Saving vector store...")
    vector_store.save_local("faiss_index")

    print("✅ Vector store created and saved!")
    return vector_store

# Create vector store
if documents:
    vector_store = create_vector_store(documents)
else:
    print("❌ No documents to process")

🔄 Creating embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔄 Creating FAISS vector store...
💾 Saving vector store...
✅ Vector store created and saved!


In [6]:
# Cell 6: Download the trained model
from google.colab import files

# Create zip of the FAISS index
!zip -r faiss_index_colab.zip faiss_index/

# Download
print("📥 Downloading FAISS index...")
files.download('faiss_index_colab.zip')

print("✅ Download complete! Use this in your local Streamlit app.")

  adding: faiss_index/ (stored 0%)
  adding: faiss_index/index.faiss (deflated 7%)
  adding: faiss_index/index.pkl (deflated 73%)
📥 Downloading FAISS index...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download complete! Use this in your local Streamlit app.
